# Approach A3: XLM-RoBERTa Full Fine-Tuning Dual Encoder
## Khmer Legal Information Retrieval — Deep Learning Final Project

This notebook trains and benchmarks **Approach A3** on the official Cambodian Civil Code (2007) and Criminal Code (2009):
- **Backbone**: `intfloat/multilingual-e5-base` (278M parameters, all layers trainable)
- **Loss**: InfoNCE with in-batch negatives ($\tau = 0.05$)
- **Optimizer**: AdamW + 10% linear warmup + Cosine Annealing decay
- **Hardware**: Google Colab T4 GPU (with `fp16` mixed precision for ~8s per epoch)
- **Benchmarks**: Primary T-Q (200 human-verified questions) and Secondary T-T (148 article titles)

### Step 1: Check GPU Acceleration
Verify that the Colab runtime is using a GPU (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/mengchheanglong/khmer-legal-retrieval.git
%cd khmer-legal-retrieval
!git checkout main
!pip uninstall -y torchvision
!pip install -q -r requirements.txt


### Step 3: Run Approach A3 Unit Tests

In [ ]:
!pytest tests/unit/test_a3_xlmr_finetune.py -v

### Step 4: Execute 6-Configuration Hyperparameter Grid Search
Grid: Learning Rate $\in \{1\times 10^{-5}, 2\times 10^{-5}, 3\times 10^{-5}\} \times \text{Weight Decay} \in \{0.0, 0.01\}$.
On a T4 GPU with `fp16`, each epoch completes in ~8 seconds (~40 seconds per run, ~4.5 minutes total).

In [ ]:
# Option A (Recommended for Colab T4 GPU): Full 6-config grid (~15 min total)
!python -m src.dl.experiments.tune_a3 --grid full --epochs 5 --batch-size 16 --patience 3

# Option B (Fastest): 4-config 2x2 grid (~10 min total)
# !python -m src.dl.experiments.tune_a3 --grid 2x2 --epochs 5 --batch-size 16 --patience 3


### Step 5: Display Hyperparameter Tuning Summary Table

In [ ]:
import pandas as pd
df_a3 = pd.read_csv('results/tuning/a3.csv')
print(df_a3.to_string(index=False))

### Step 6: Plot Training Dynamics
Plot per-epoch training loss and validation MRR@10 curves for each grid configuration.

In [ ]:
import glob
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for f in sorted(glob.glob('results/logs/a3_*.csv')):
    name = f.split('/')[-1].replace('.csv', '').replace('a3_', '')
    df_run = pd.read_csv(f)
    ax1.plot(df_run['epoch'], df_run['train_loss'], marker='o', label=name)
    ax2.plot(df_run['epoch'], df_run['val_mrr10'], marker='s', label=name)

ax1.set_title('Training Loss (InfoNCE)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.set_title('Validation MRR@10')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MRR@10')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

### Step 7: View Final Benchmark Results on Full 1,976-Article Corpus

In [ ]:
import json
with open('results/metrics/a3_xlmr.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print("=== Primary Benchmark (T-Q: 200 Questions) ===")
for metric, vals in results['tq_benchmark']['overall'].items():
    print(f"{metric:10s}: {vals['mean']:.4f} [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")

print("\n=== Secondary Benchmark (T-T: 148 Titles) ===")
for metric, vals in results['tt_benchmark']['overall'].items():
    print(f"{metric:10s}: {vals['mean']:.4f} [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")

### Step 8: Regenerate Error Analysis, Reports & Visualizations
Automatically recomputes error taxonomy rankings, statistical significance tests, and publication figures.

In [ ]:
!python -m src.dl.error_analysis --models all
!python -m src.dl.report
!python -m src.dl.visualize


### Step 9: Package & Download All Updated Artifacts
Download a zip containing all new telemetry, checkpoints, tables, and figures to place into your local repository.

In [ ]:
!zip -r a3_results.zip results/tuning/a3.csv results/metrics/ results/logs/a3_* results/checkpoints/a3_xlmr_finetune/best.pt results/error_analysis/ results/figures/
from google.colab import files
files.download('a3_results.zip')
